# Imports

In [ ]:
import torch
import torch.nn as nn

from torchvision import datasets
from torchvision import transforms

# Dataset

In [ ]:
transform = transforms.ToTensor()

dataset = datasets.CIFAR10(
    root="./data",
    train=True,
    download=False,
    transform=transform
)

c:\Users\dhana\miniconda3\envs\DL_Torch\Lib\site-packages\torchvision\datasets\cifar.py:83: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  entry = pickle.load(f, encoding="latin1")


# Get Two Images

In [ ]:
image1, _ = dataset[0]
image2, _ = dataset[1]

# Add batch dimension: 

image1 = image1.unsqueeze(0)
image2 = image2.unsqueeze(0)

print(image1.shape)

torch.Size([1, 3, 32, 32])


# Encoder

In [ ]:
class Encoder(nn.Module):

    def __init__(self):
        super().__init__()

        self.encoder = nn.Sequential(

            nn.Conv2d(
                in_channels=3,
                out_channels=16,
                kernel_size=3,
                padding=1
            ),
            nn.ReLU(),

            nn.MaxPool2d(2),

            nn.Conv2d(
                in_channels=16,
                out_channels=32,
                kernel_size=3,
                padding=1
            ),
            nn.ReLU(),

            nn.MaxPool2d(2),

            nn.Flatten(),

            nn.Linear(
                32 * 8 * 8,
                128
            )
        )

    def forward(self, x):
        return self.encoder(x)

# Predictor

In [ ]:
class Predictor(nn.Module):

    def __init__(self):
        super().__init__()

        self.predictor = nn.Sequential(

            nn.Linear(128, 256),

            nn.ReLU(),

            nn.Linear(256, 128)
        )

    def forward(self, x):
        return self.predictor(x)

# Create Models

In [ ]:
online_encoder = Encoder()

target_encoder = Encoder()

predictor = Predictor()

# Copy Weights

In [ ]:
target_encoder.load_state_dict(
    online_encoder.state_dict()
)

<All keys matched successfully>

# Freeze Target Encoder

In [ ]:
for param in target_encoder.parameters():
    param.requires_grad = False

# Loss Function

In [ ]:
loss_fn = nn.MSELoss()

# Optimizer

In [ ]:
optimizer = torch.optim.Adam(
    list(online_encoder.parameters()) +
    list(predictor.parameters()),
    lr=0.001
)

# Forward Pass

### Online side

In [ ]:
z1 = online_encoder(image1)

pred_z2 = predictor(z1)

### Target side

In [ ]:
with torch.no_grad():

    z2 = target_encoder(image2)

In [ ]:
print(z1.shape)
print(pred_z2.shape)
print(z2.shape)

torch.Size([1, 128])
torch.Size([1, 128])
torch.Size([1, 128])


# Compute Loss

In [ ]:
loss = loss_fn(
    pred_z2,
    z2
)

print(loss)

tensor(0.0072, grad_fn=<MseLossBackward0>)


# Backpropagation

In [ ]:
optimizer.zero_grad()
loss.backward()
optimizer.step()

Online Encoder  ✓ updated

Predictor       ✓ updated

Target Encoder  ✗ unchanged

# EMA Update

In [ ]:
m = 0.99

for target_param, online_param in zip(
    target_encoder.parameters(),
    online_encoder.parameters()
):

    target_param.data = (
        m * target_param.data
        +
        (1 - m) * online_param.data
    )

# Full Architecture

image 1 -> online Encoder -> z1 -> predictor -> pred_z2

image 2 -> target Encoder -> z2

MSE loss

# Training Loop

In [ ]:
for epoch in range(100):
    
    # find pred z2 using z1
    z1 = online_encoder(image1)
    pred_z2 = predictor(z1)

    # find actual z2
    with torch.no_grad():
        z2 = target_encoder(image2)
    
    # find loss
    loss = loss_fn(z2, pred_z2)

    # This update online_encoder
    optimizer.zero_grad() # Clear the old math.
    loss.backward() # This function performs the actual backpropagation, but it only does the math.
    optimizer.step() # This is where the actual updating happens.

    # update target_encoder
    for target_param, online_param in zip(
        target_encoder.parameters(),
        online_encoder.parameters()
    ):
        
        # formula to update target_encoder a small step
        # like .2 .2 .2
        target_param.data = (m * target_param) + ((1-m) * online_param)

    print(epoch, loss.item())



0 0.0038131794426590204
1 0.0024223399814218283
2 0.0011798832565546036
3 0.0005441568791866302
4 0.0003747592563740909
5 0.0003252199094276875
6 0.00030614802381023765
7 0.00027983516338281333
8 0.00024116034910548478
9 0.0002151913067791611
10 0.000186384393600747
11 0.0001484920212533325
12 0.00012564797361847013
13 0.000105006416561082
14 8.264800999313593e-05
15 7.077045302139595e-05
16 5.867323852726258e-05
17 4.646527304430492e-05
18 4.027817340102047e-05
19 3.2219999411609024e-05
20 2.559909989940934e-05
21 2.4339415176655166e-05
22 2.2341544536175206e-05
23 2.1455784008139744e-05
24 2.2680193069390953e-05
25 2.136397779395338e-05
26 2.058541213045828e-05
27 2.0166145986877382e-05
28 1.789809175534174e-05
29 1.6349767975043505e-05
30 1.4518970601784531e-05
31 1.1644620826700702e-05
32 9.827199392020702e-06
33 8.071961929090321e-06
34 6.308455795078771e-06
35 5.653804691974074e-06
36 4.851073754252866e-06
37 4.211488430883037e-06
38 4.22346329287393e-06
39 3.878653842548374e-06


In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10,3))

plt.subplot(1,3,1)
plt.imshow(image1)
plt.title("Original image1")

plt.subplot(1,3,1)
plt.imshow(image2)
plt.title("Original image2")

plt.subplot(1,3,2)
plt.imshow(z1.squeeze(0).permute(1,2,0))
plt.title("View1")

plt.subplot(1,3,2)
plt.imshow(pred_z2.squeeze(0).permute(1,2,0))
plt.title("View1")

plt.subplot(1,3,3)
plt.imshow(z2.squeeze(0).permute(1,2,0))
plt.title("View2")

plt.show()